#  أولاً: Computer Vision (CV)

##  1- ResNet 
- **ResNet50**
- **ResNet101**
- **ResNet152**
كان وما زال أشهر موديل للـ feature extraction.

---

## 2- VGG (قديم بس لسه مستخدم)
- **VGG16**
- **VGG19**
سهل وبسيط وممتاز للـ transfer learning.

---

## 3- Inception Models
- **InceptionV3**
- **Inception-ResNet**
قوية جدًا في مهام التصنيف.

---

## 4- EfficientNet
- **EfficientNet-B0 → B7**
نماذج فعّالة جدًا ودقيقة + قليلة الحسابات.

---

## 5- MobileNet
- **MobileNetV2**
- **MobileNetV3**
ممتازة للهواتف والـ embedded devices.

---

## 6- DenseNet
- **DenseNet121**
- **DenseNet169**
شبكة كثيفة وممتازة في نقل الخصائص.

---

## 7- Vision Transformers (ViT)
- **ViT-B/16**
- **ViT-L/16**
أصبحت أكثر انتشارًا حاليًا في المشاريع الحديثة.

---

#  ثانيًا: NLP (Natural Language Processing)

## 1- BERT (الأشهر على الإطلاق)
- **BERT-base**
- **BERT-large**
- **DistilBERT** (نسخة خفيفة)

---

## 2- GPT Models
- **GPT-2**
- **GPT-3** (غير مفتوح بالكامل لكنها فكرة الـ pretrained LM)

---

## 3- RoBERTa
تحسين لـ BERT — ممتاز في معظم المهام.

---

## 4- XLM-R
لنقل المعرفة بين لغات مختلفة (Cross-Lingual).

---

## 5- T5
موديل موحد لكل مهام NLP.

---

## 6- ALBERT
نسخة أخف وأسرع من BERT.

---

# 🔵 ثالثًا: Audio Models

## 1️⃣ Wav2Vec 2.0
من Facebook — ممتاز في speech recognition.

---

## 2️⃣ Whisper
من OpenAI — أقوى موديل للترجمة والتفريغ الصوتي.


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

IMG_SIZE = (224, 224)   # مناسب لـ VGG & ResNet
BATCH = 64

# Resize + Normalize
def preprocess(x, y, train=False):
    x = tf.image.resize(x, IMG_SIZE)
    x = tf.cast(x, tf.float32)/255.0
    if train:
        x = tf.image.random_flip_left_right(x)
    return x, y

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000)
    .map(lambda x,y: preprocess(x,y,True))
    .batch(BATCH).prefetch(2)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(lambda x,y: preprocess(x,y,False))
    .batch(BATCH).prefetch(2)
)


In [5]:
def build_scratch():
    inputs = layers.Input(shape=(224,224,3))
    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    return models.Model(inputs, outputs)

model_scratch = build_scratch()
model_scratch.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model_scratch.fit(train_ds, epochs=3, validation_data=test_ds)
model_scratch.save("cnn_scratch.h5")


Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 62ms/step - accuracy: 0.3683 - loss: 2.0939 - val_accuracy: 0.4900 - val_loss: 1.4179
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 45s 57ms/step - accuracy: 0.5349 - loss: 1.3049 - val_accuracy: 0.5507 - val_loss: 1.2709
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 45s 57ms/step - accuracy: 0.5965 - loss: 1.1396 - val_accuracy: 0.6049 - val_loss: 1.1087


In [6]:
base = tf.keras.applications.VGG16(
    include_top=False, weights="imagenet", input_shape=(224,224,3), pooling="avg"
)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [7]:
base.trainable = False   # تجميد الموديل freezing

inputs = layers.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)
model_tl = models.Model(inputs, outputs)

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_tl.fit(train_ds, epochs=3, validation_data=test_ds)
model_tl.save("model_frozen.h5")


Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 129s 145ms/step - accuracy: 0.2299 - loss: 2.1511 - val_accuracy: 0.4365 - val_loss: 1.7615
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 108s 138ms/step - accuracy: 0.4089 - loss: 1.7492 - val_accuracy: 0.4855 - val_loss: 1.6181
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 108s 138ms/step - accuracy: 0.4350 - loss: 1.6504 - val_accuracy: 0.5006 - val_loss: 1.5430


In [8]:
# نفك آخر 30% من الليرات
base.trainable = True # no freezing unless 30%  

fine_tune_at = int(len(base.layers) * 0.7)

for i, layer in enumerate(base.layers):
    layer.trainable = i >= fine_tune_at

model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR صغير جداً
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_tl.fit(train_ds, epochs=10, validation_data=test_ds)
model_tl.save("model_finetuned.h5")


Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 132s 163ms/step - accuracy: 0.5527 - loss: 1.2833 - val_accuracy: 0.7092 - val_loss: 0.8538
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 126s 160ms/step - accuracy: 0.6938 - loss: 0.8813 - val_accuracy: 0.7402 - val_loss: 0.7485
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.7366 - loss: 0.7577 - val_accuracy: 0.7618 - val_loss: 0.6869
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.7638 - loss: 0.6861 - val_accuracy: 0.7844 - val_loss: 0.6219
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.7795 - loss: 0.6358 - val_accuracy: 0.7945 - val_loss: 0.5997
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.7962 - loss: 0.5908 - val_accuracy: 0.7831 - val_loss: 0.6122
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.8050 - loss: 0.5587 - val_accuracy: 0.8120 - val_loss: 0.5400
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 125s 160ms/step - accuracy: 0.8184 -